# Load Raw BBC Dataset

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data")

rows = []

for label_dir in data_dir.iterdir():
    if label_dir.is_dir():
        label = label_dir.name

        for file_path in label_dir.glob("*.txt"):
            text = file_path.read_text(encoding="latin-1")
            rows.append({
                "text": text,
                "label": label
            })

df = pd.DataFrame(rows)

print(df.head())
print(df["label"].value_counts())

                                                text          label
0  Musicians to tackle US red tape\n\nMusicians' ...  entertainment
1  U2's desire to be number one\n\nU2, who have w...  entertainment
2  Rocker Doherty in on-stage fight\n\nRock singe...  entertainment
3  Snicket tops US box office chart\n\nThe film a...  entertainment
4  Ocean's Twelve raids box office\n\nOcean's Twe...  entertainment
label
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


# Clean text Data

In [2]:
df = df.dropna(subset=["text", "label"])

df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]

print(df.shape)

(2225, 2)


# Encode Label

In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["label_id"] = label_encoder.fit_transform(df["label"])

print(label_encoder.classes_)
print(df[["label", "label_id"]].head())

['business' 'entertainment' 'politics' 'sport' 'tech']
           label  label_id
0  entertainment         1
1  entertainment         1
2  entertainment         1
3  entertainment         1
4  entertainment         1


# Train/Validation/Test Split

In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label_id"]
)

print(len(train_df), len(val_df), len(test_df))

1780 222 223


# Initialize BertTokenizer

In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/opt/anaconda3/envs/ml_conda_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Testing on 1 text

In [6]:
sample_text = train_df.iloc[0]["text"]

encoding = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

print(encoding.keys())
print(encoding["input_ids"].shape)
print(encoding["attention_mask"].shape)
print(encoding["token_type_ids"].shape)

KeysView({'input_ids': tensor([[  101, 11348,  2452,  5495,  7237,  2000,  3577, 11348,  1005,  1055,
          2002,  3170,  7520,  2452,  4284,  1011,  2345,  5495,  2114, 12170,
          2906, 14778,  2480,  2006,  1017,  2258,  2038,  2042,  7237,  2000,
          2613, 27084,  6340,  4215,  1005,  1055, 14674,  8780,  2139,  2019,
          8913,  2696,  3346,  1999,  2624,  6417,  1012,  2613,  1005,  1055,
          2598,  4324,  3590,  1010,  2199,  6168,  1996, 27985,  4078,  2998,
         12943, 19231,  6906,  1999, 12170,  2906, 14778,  2480,  2038,  1037,
          3977,  1997,  2074,  2260,  1010,  5764,  2581,  1012,  1996,  3493,
          2874,  2097,  2022,  2445,  2012,  2560,  1022,  1010,  2199,  9735,
          1012,  1000,  1996,  3247,  2000,  2693,  2001,  1037,  3697,  2028,
          1010,  2021,  2004,  2057,  2641,  1996,  4599,  2004,  2028,  1997,
          2256,  3078, 11100,  1010,  1000,  2056, 12170,  2906, 14778,  2480,
          3472, 13389,  3235,

In [7]:
input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]
token_type_ids = encoding["token_type_ids"]

# Building Pytorch Dataset

In [8]:
import torch
from torch.utils.data import Dataset

class BBCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding["token_type_ids"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Split Train/Val/Test Dataset

In [9]:
train_dataset = BBCDataset(
    train_df["text"],
    train_df["label_id"],
    tokenizer,
    max_length=256
)

val_dataset = BBCDataset(
    val_df["text"],
    val_df["label_id"],
    tokenizer,
    max_length=256
)

test_dataset = BBCDataset(
    test_df["text"],
    test_df["label_id"],
    tokenizer,
    max_length=256
)

# Buidl DataLoader

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# Veify Batch Output

In [12]:
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("token_type_ids:", batch["token_type_ids"].shape)
print("labels:", batch["labels"].shape)

input_ids: torch.Size([16, 256])
attention_mask: torch.Size([16, 256])
token_type_ids: torch.Size([16, 256])
labels: torch.Size([16])
